# RAG System – Austrian Tax Law

**Model:** Mistral-7B-Instruct-v0.3 (4-bit quantized)
**Retrieval:** ChromaDB with paragraph-based chunking

---

## How does the model work?

This system uses Retrieval-Augmented Generation (RAG), combining two components:

1. **Document loading** – Austrian tax law texts (RTF) are read and converted to plain text
2. **Chunking** – Texts are split at paragraph markers (§) to preserve legal context
3. **Vectorization** – ChromaDB stores each chunk as a vector (embedding)
4. **Retrieval** – For each question, the most semantically relevant paragraphs are retrieved
5. **Generation** – Mistral 7B answers the question based solely on the retrieved paragraphs

---

## Setup

1. **Enable GPU:** Runtime -> Change runtime type -> T4 GPU
2. **Google Drive:** Place your law RTF files in a folder on Google Drive
3. **Dataset:** Download dataset_clean.csv from GitHub
4. **Adjust paths:** Only in Cell 2 (Configuration)
5. **Run:** Runtime -> Run allSonnet 4.6

---

## Data Sources

The legal documents used for retrieval were sourced from the Austrian Legal
Information System Ris -> ris.bka.gv.at.

The following tax laws were downloaded as RTF files and used as the
knowledge base for the retrieval system:

- **EStG** (Einkommensteuergesetz) – Income Tax Act: § 1, 22, 23, 23a, 25, 26, 27a, 27b, 28, 32, 33
- **KStG** (Körperschaftsteuergesetz) – Corporate Income Tax Act: § 7, 8, 9, 10, 11, 12, 12a
- **UStG** (Umsatzsteuergesetz) – Value Added Tax Act: § 1–10, 3a
- **GrEStG** (Grunderwerbsteuergesetz) – Real Estate Transfer Tax Act

Only the paragraphs most relevant to the questions in the dataset were
selected to keep the knowledge base focused and retrieval precise.

## Cell 1 – Dependencies
Installation of required libraries:
- transformers: loading and running language models (Mistral)
- chromadb: vector database for retrieval
- sentence-transformers: converting text to vectors (embeddings)
- striprtf: converting RTF files to plain text
- bitsandbytes: 4-bit quantization to reduce VRAM usage
- accelerate: automatic distribution of the model across available hardware

In [ ]:
!pip install -q transformers chromadb sentence-transformers striprtf pandas torch accelerate bitsandbytes

## Cell 2 – Configuration
Central configuration of all file paths.
This cell needs to be adjusted to run the notebook on a different system. Google Drive is mounted to access the law texts and dataset and to save the results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ↓ Pfad zum Ordner mit deinen Gesetze-RTF-Dateien
GESETZE_ORDNER = '/content/drive/MyDrive/Colab Notebooks/Gesetze/'

# ↓ Pfad zur dataset_clean.csv
DATASET_PFAD = '/content/drive/MyDrive/Colab Notebooks/dataset_clean.csv'

# ↓ Pfad wo die Output-CSV gespeichert wird
OUTPUT_PFAD = '/content/drive/MyDrive/Colab Notebooks/output_rag_mistral.csv'

# Prüfen ob alles vorhanden:
print('Gesetze-Dateien gefunden:')
rtf_dateien = [f for f in os.listdir(GESETZE_ORDNER) if f.endswith('.rtf')]
for f in sorted(rtf_dateien):
    print(f'   {f}')
print(f'\n {len(rtf_dateien)} RTF-Dateien gefunden')
print(f'Dataset vorhanden: {os.path.exists(DATASET_PFAD)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Gesetze-Dateien gefunden:
   ESTG 1.rtf
   ESTG 22.rtf
   ESTG 23.rtf
   ESTG 23a.rtf
   ESTG 25.rtf
   ESTG 26.rtf
   ESTG 27a.rtf
   ESTG 27b.rtf
   ESTG 28.rtf
   ESTG 32.rtf
   ESTG 33.rtf
   GRESTG.rtf
   KSTG 10.rtf
   KSTG 11.rtf
   KSTG 12.rtf
   KSTG 12a.rtf
   KSTG 7.rtf
   KSTG 8.rtf
   KSTG 9.rtf
   USTG 1.rtf
   USTG 10.rtf
   USTG 2.rtf
   USTG 3.rtf
   USTG 3a.rtf
   USTG 4.rtf
   USTG 5.rtf
   USTG 6.rtf
   USTG 7.rtf
   USTG 8.rtf
   USTG 9.rtf

 30 RTF-Dateien gefunden
Dataset vorhanden: True


## Cell 3 – Load law texts
All RTF files in the laws folder are read and converted to plain text.
The function rtf_to_text removes all RTF formatting codes.
Whitespace is normalized to ensure clean further processing.

In [ ]:
from striprtf.striprtf import rtf_to_text

def gesetze_laden(ordner):
    gesetze = []
    for datei in sorted(os.listdir(ordner)):
        if datei.endswith('.rtf'):
            pfad = os.path.join(ordner, datei)
            with open(pfad, 'r', encoding='utf-8', errors='ignore') as f:
                text = rtf_to_text(f.read())
                text = ' '.join(text.split())  # Whitespace normalisieren
                gesetze.append({'name': datei, 'text': text})
                print(f' Geladen: {datei}')
    print(f'\n {len(gesetze)} Dateien geladen!')
    return gesetze

gesetze = gesetze_laden(GESETZE_ORDNER)

 Geladen: ESTG 1.rtf
 Geladen: ESTG 22.rtf
 Geladen: ESTG 23.rtf
 Geladen: ESTG 23a.rtf
 Geladen: ESTG 25.rtf
 Geladen: ESTG 26.rtf
 Geladen: ESTG 27a.rtf
 Geladen: ESTG 27b.rtf
 Geladen: ESTG 28.rtf
 Geladen: ESTG 32.rtf
 Geladen: ESTG 33.rtf
 Geladen: GRESTG.rtf
 Geladen: KSTG 10.rtf
 Geladen: KSTG 11.rtf
 Geladen: KSTG 12.rtf
 Geladen: KSTG 12a.rtf
 Geladen: KSTG 7.rtf
 Geladen: KSTG 8.rtf
 Geladen: KSTG 9.rtf
 Geladen: USTG 1.rtf
 Geladen: USTG 10.rtf
 Geladen: USTG 2.rtf
 Geladen: USTG 3.rtf
 Geladen: USTG 3a.rtf
 Geladen: USTG 4.rtf
 Geladen: USTG 5.rtf
 Geladen: USTG 6.rtf
 Geladen: USTG 7.rtf
 Geladen: USTG 8.rtf
 Geladen: USTG 9.rtf

 30 Dateien geladen!


## Cell 4 – Build ChromaDB vector database
Paragraph-based chunking:
The text is split at paragraph markers (§) so that each paragraph remains
intact as a chunk. Paragraphs exceeding 300 words are further split with
an overlap of 50 words to avoid losing content at chunk boundaries.

Vectorization:
Each chunk is automatically converted into a vector (embedding) by ChromaDB.
Queries find the semantically most similar chunks regardless of exact keyword matches.

In [ ]:
import chromadb, re

def in_chunks_paragraph(text, max_woerter=300, overlap_woerter=50):
    """
    Splittet Text zuerst nach Paragraphen (§), dann nach Wörtern wenn zu groß.
    So bleibt jeder § als zusammenhängender Kontext erhalten.
    """
    # Nach § splitten
    paragraphen = re.split(r'(?=§\s*\d)', text)
    paragraphen = [p.strip() for p in paragraphen if p.strip()]

    chunks = []
    for para in paragraphen:
        woerter = para.split()
        if len(woerter) <= max_woerter:
            # Paragraph passt in einen Chunk
            chunks.append(para)
        else:
            # Zu langer Paragraph → mit Overlap aufteilen
            i = 0
            while i < len(woerter):
                chunk = ' '.join(woerter[i:i+max_woerter])
                chunks.append(chunk)
                i += max_woerter - overlap_woerter
    return chunks

# ChromaDB neu aufbauen
client = chromadb.Client()
try:
    client.delete_collection('steuerrecht')
except:
    pass

sammlung = client.create_collection('steuerrecht')

chunk_id = 0
for gesetz in gesetze:
    chunks = in_chunks_paragraph(gesetz['text'])
    for chunk in chunks:
        sammlung.add(
            documents=[chunk],
            ids=[f'chunk_{chunk_id}'],
            metadatas=[{'quelle': gesetz['name']}]
        )
        chunk_id += 1

print(f'ChromaDB befüllt mit {chunk_id} Chunks!')

ChromaDB befüllt mit 637 Chunks!


## Cell 5 – Load Mistral 7B
The model Mistral-7B-Instruct-v0.3 is loaded with 4-bit quantization.
This reduces VRAM requirements from ~14 GB to ~5 GB, making it possible
to run on the free T4 GPU in Google Colab. Loading takes approximately
2-3 minutes. No HuggingFace token required.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

modell_name = 'mistralai/Mistral-7B-Instruct-v0.3'

# 4-bit Quantisierung: reduziert VRAM von ~14GB auf ~5GB
bnb_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(modell_name)
tokenizer.pad_token = tokenizer.eos_token

modell = AutoModelForCausalLM.from_pretrained(
    modell_name,
    quantization_config=bnb_config,
    device_map='auto'  # automatisch auf GPU
)
print('Mistral 7B geladen!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral 7B geladen!


## Cell 6 – RAG function
The function frage_beantworten implements the RAG pipeline in five steps:
1. The 3 most semantically similar paragraphs are retrieved from ChromaDB
2. The retrieved paragraphs are embedded as context into a prompt
3. The prompt is tokenized and truncated to a maximum of 2048 tokens
4. Mistral generates an answer based solely on the provided context
5. Only the newly generated tokens are decoded and returned

Parameters: do_sample=False ensures deterministic outputs,
repetition_penalty=1.2 prevents repetitive text.

In [ ]:
def frage_beantworten(frage):
    # Schritt 1: Relevante Paragraphen aus ChromaDB abrufen
    ergebnisse = sammlung.query(query_texts=[frage], n_results=5)
    kontext = ' '.join(ergebnisse['documents'][0])

    # Schritt 2: Prompt im Mistral Chat-Format aufbauen
    messages = [
        {
            "role": "system",
            "content": (
                "Du bist ein Experte für österreichisches Steuerrecht. "
                "Beantworte die Frage kurz in 1 oder maximal 2 Sätzen NUR basierend auf dem Gesetzestext. "
                "Nenne immer § und Gesetz wenn möglich. "
                "Wenn die Antwort nicht im Text steht: 'Nicht im Gesetzestext gefunden.'"
            )
        },
        {
            "role": "user",
            "content": f"Gesetzestext:\n{kontext}\n\nFrage: {frage}"
        }
    ]

    # Schritt 3: Prompt tokenisieren
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=2048).to(device)

    # Schritt 4: Antwort generieren
    outputs = modell.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )

    # Schritt 5: Nur die neue Antwort extrahieren (ohne Prompt)
    input_len = inputs['input_ids'].shape[1]
    antwort_tokens = outputs[0][input_len:]
    antwort = tokenizer.decode(antwort_tokens, skip_special_tokens=True).strip()
    return antwort

## Cell 7 – Test with 10 questions
Optional test run with the first 10 questions for quality control
before the full run. The cell is commented out and can be enabled if needed.

In [ ]:
#import pandas as pd

#df = pd.read_csv(DATASET_PFAD)
#df_test = df.head(10)

#for i, zeile in df_test.iterrows():
 #   try:
  #      antwort = frage_beantworten(zeile['prompt'])
   #     print(f'✅ Frage {i+1}: {zeile["prompt"]}')
    #    print(f'   Antwort: {antwort}')
     #   print('---')
    #except Exception as e:
     #   print(f' Fehler bei Frage {i+1}: {e}')

#print(' Test fertig! Wenn Antworten gut aussehen → Zelle 8 ausführen')

## Cell 8 – Answer all 643 questions
All questions in the dataset are answered and saved as a CSV file.
Results are written to Google Drive every 10 questions. If the session
is interrupted, the cell automatically detects which questions have already
been answered and resumes from that point. Estimated runtime: 90-120 minutes on a T4 GPU.

In [ ]:
import pandas as pd, os

df = pd.read_csv(DATASET_PFAD)
print(f'📋 {len(df)} Fragen geladen.')

# Bereits gespeicherte Antworten laden (falls Session abgebrochen)
if os.path.exists(OUTPUT_PFAD):
    df_done = pd.read_csv(OUTPUT_PFAD)
    done_ids = set(df_done['id'].tolist())
    ergebnisse = df_done.to_dict('records')
    print(f' {len(done_ids)} bereits fertig, weiter ab da...')
else:
    done_ids = set()
    ergebnisse = []

# Alle Fragen beantworten
for i, zeile in df.iterrows():
    if zeile['id'] in done_ids:
        continue

    try:
        antwort = frage_beantworten(zeile['prompt'])
        ergebnisse.append({'id': zeile['id'], 'answer': antwort})
    except Exception as e:
        ergebnisse.append({'id': zeile['id'], 'answer': ''})
        print(f' Fehler bei {zeile["id"]}: {e}')

    # Alle 10 Fragen speichern + Fortschritt anzeigen
    if len(ergebnisse) % 10 == 0:
        pd.DataFrame(ergebnisse).to_csv(OUTPUT_PFAD, index=False)
        print(f' {len(ergebnisse)}/{len(df)} ({len(ergebnisse)/len(df)*100:.1f}%)')

# Finales Speichern
pd.DataFrame(ergebnisse).to_csv(OUTPUT_PFAD, index=False)
print(f'\n Fertig! {len(ergebnisse)} Antworten gespeichert in:')
print(OUTPUT_PFAD)

📋 643 Fragen geladen.
 310 bereits fertig, weiter ab da...
 320/643 (49.8%)
 330/643 (51.3%)
 340/643 (52.9%)
 350/643 (54.4%)
 360/643 (56.0%)
 370/643 (57.5%)
 380/643 (59.1%)
 390/643 (60.7%)
 400/643 (62.2%)
 410/643 (63.8%)
 420/643 (65.3%)
 430/643 (66.9%)
 440/643 (68.4%)
 450/643 (70.0%)
 460/643 (71.5%)
 470/643 (73.1%)
 480/643 (74.7%)
 490/643 (76.2%)
 500/643 (77.8%)
 510/643 (79.3%)
 520/643 (80.9%)
 530/643 (82.4%)
 540/643 (84.0%)
 550/643 (85.5%)
 560/643 (87.1%)
 570/643 (88.6%)
 580/643 (90.2%)
 590/643 (91.8%)
 600/643 (93.3%)
 610/643 (94.9%)
 620/643 (96.4%)
 630/643 (98.0%)
 640/643 (99.5%)

 Fertig! 643 Antworten gespeichert in:
/content/drive/MyDrive/Colab Notebooks/output_rag_mistral.csv


Cell 9 – Verify output
Checks the saved CSV for correct format: 643 rows, columns id and answer.

In [ ]:
import pandas as pd

df_check = pd.read_csv(OUTPUT_PFAD)
print(f' {len(df_check)} Zeilen in der Output-CSV')
print(f' Spalten: {list(df_check.columns)}')
print(f'\nErste 3 Antworten zur Kontrolle:')
print('---')
for _, zeile in df_check.head(3).iterrows():
    print(f'ID: {zeile["id"]}')
    print(f'Antwort: {zeile["answer"]}')
    print('---')

 643 Zeilen in der Output-CSV
 Spalten: ['id', 'answer']

Erste 3 Antworten zur Kontrolle:
---
ID: CORP-TAX-001
Antwort: Die Bemessungsgrundlage für die Körperschaftsteuer ist der gesamte Umsatz (§ 2 Abs. 8 des Einkommensteuergesetzes 1988).
---
ID: CORP-TAX-002
Antwort: In Österreich gibt es keinen expliziten Gesetzestext, der speziell die Steuerkonsequenz einer Zinslosgeldeserhöhung durch eine Kapitalgesellschaft beschreibt. Allgemeine Bestimmungen zum Steuernachweis von Zinsen finden sich in § 10a Abs. 4 Z 2 EStG. Daher liegen keine direkten Steuerkonsequenzen vor, falls eine Kap
---
ID: CORP-TAX-003
Antwort: Inländische Körperschaften, die sich einer Unternehmensgruppe nach § 47a EStDV (Gewerbebetriebe) zufolge angehören, müssen ihre Einkünfte den Einkünften aus Gewerbebetrieb zurechnen.
---


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import json

dateien = [
    '/content/drive/MyDrive/Colab Notebooks/ABGABE/Evaluation_script.ipynb',
    '/content/drive/MyDrive/Colab Notebooks/ABGABE/FT_model_submission.ipynb',
    '/content/drive/MyDrive/Colab Notebooks/ABGABE/RAG_model_submission.ipynb',
]

for path in dateien:
    with open(path, 'r') as f:
        nb = json.load(f)
    if 'widgets' in nb.get('metadata', {}):
        del nb['metadata']['widgets']
    with open(path, 'w') as f:
        json.dump(nb, f, indent=1)
    print(f'Gefixt: {path}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/ABGABE/Evaluation_script.ipynb'